# Online Retail Store Analysis

## Preparing environment

### Loading up needed modules

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from pathlib import Path

### Importing data

In [2]:
df = pd.read_csv(Path("datasets") / "online_retail.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [3]:
df.sample(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
217521,555925,21992,VINTAGE PAISLEY STATIONERY SET,1,2011-06-07 17:13:00,2.46,NaN,United Kingdom
220553,556199,21400,RED PUDDING SPOON,24,2011-06-09 12:40:00,0.12,16170.0,United Kingdom
36023,539437,35653,VINTAGE BEAD NOTEBOOK,1,2010-12-17 14:54:00,5.91,NaN,United Kingdom
213392,555539,85152,HAND OVER THE CHOCOLATE SIGN,4,2011-06-05 12:44:00,2.10,14472.0,United Kingdom
505738,579098,21850,BLUE DIAMANTE PEN IN GIFT BOX,1,2011-11-28 11:18:00,4.95,14606.0,United Kingdom
417758,572669,23000,TRAVEL CARD WALLET TRANSPORT,2,2011-10-25 13:10:00,0.42,16549.0,United Kingdom
20869,538071,21983,PACK OF 12 BLUE PAISLEY TISSUES,3,2010-12-09 14:09:00,0.85,NaN,United Kingdom
507867,579187,23609,SET 10 CARDS SNOWY ROBIN 17099,1,2011-11-28 15:31:00,2.91,NaN,United Kingdom
138495,548198,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2011-03-29 16:05:00,4.95,16712.0,United Kingdom
128867,547362,22505,MEMO BOARD COTTAGE DESIGN,24,2011-03-22 12:33:00,4.25,13694.0,United Kingdom


# Data preparation

## Checking column completeness

In [ ]:
all_cols = df.columns
bar_colors = []
counted_cols = []
for col_name in all_cols:
    completeness = 1 - len(df.loc[df[col_name].isnull(), col_name]) / len(df)
    counted_cols.append(completeness)
    if completeness == 1.0:
        bar_colors.append("darkgreen")
    elif completeness > 0.8:
        bar_colors.append("gold")
    else:
        bar_colors.append("red")


def percentage_format(x, position):
    return f"{round(x * 100)}%"


fig, axs = plt.subplots(figsize=(len(counted_cols) * 2, 5))
axs.set_title("Data completeness %")
axs.yaxis.set_major_formatter(percentage_format)
axs.bar(all_cols, counted_cols, color=bar_colors)
axs.set_ylim(0, 1)
plt.show()

### Missing Customer Data

As we can see, the dataset lacks `CustomerID` for many records. By dropping these rows, this analysis excludes "guest checkouts". Our focus is strictly on registered users, allowing for a genuine evaluation of customer loyalty, retention metrics, and long-term user behavior.

### Data Architecture: Splitting the Dataset

To ensure absolute metric accuracy, we must branch our dataset based on the business question being asked:

1. **Net Revenue & LTV (Full Dataset):** The original dataset includes invoices starting with "C" (cancellations and returns). These are crucial for calculating true Net Lifetime Value (LTV), as returns offset gross revenue.
2. **Cohort Retention (Filtered Dataset):** For analyzing user loyalty, we create a separate `df_no_cancel` dataframe excluding cancellations. If we kept returns here, our grouping logic would register a product return in a subsequent month as a new "active shopping session," artificially inflating our retention rates.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df = df.assign(TotalPrice=lambda x: x["UnitPrice"] * x["Quantity"])

In [6]:
df_no_cancel = df.dropna(subset="CustomerID")

# Removing cancelled invoices and returned products
df_no_cancel = df_no_cancel[~df_no_cancel["InvoiceNo"].str.startswith("C")]
df_no_cancel = df_no_cancel[
    (df_no_cancel["Quantity"] > 0) & (df_no_cancel["UnitPrice"] > 0)
]
df_no_cancel.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## Cohort retention analysis

Here I am assigning new columns to the retention data frame:
- `InvoiceMonth` is as the name says - month an invoice was issued.
- `CohortMonth` is based on the newly created `InvoiceMonth` column - it assigns every customer to the first month they made a purchase in our store.
- `CohortIndex` is the difference in months between the first purchase and later purchases.

In [ ]:
df_retention = df_no_cancel.assign(
    InvoiceMonth=lambda x: x["InvoiceDate"].values.astype("datetime64[M]"),
    CohortMonth=lambda x: x.groupby("CustomerID")["InvoiceMonth"].transform("min"),
    CohortIndex=lambda x: (
        (x["InvoiceMonth"].dt.year - x["CohortMonth"].dt.year) * 12
        + (x["InvoiceMonth"].dt.month - x["CohortMonth"].dt.month)
    ),
)
df_retention.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,InvoiceMonth,CohortMonth,CohortIndex
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12-01,2010-12-01,0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,2010-12-01,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12-01,2010-12-01,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,2010-12-01,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,2010-12-01,0


Group by cohort months (First purchases of customers)

In [8]:
cohort_data = (
    df_retention.groupby(["CohortMonth", "CohortIndex"])
    .agg({"CustomerID": "nunique"})
    .reset_index()
)
cohort_data

,CohortMonth,CohortIndex,CustomerID
0,2010-12-01,0,885
1,2010-12-01,1,324
2,2010-12-01,2,286
3,2010-12-01,3,340
4,2010-12-01,4,321
...,...,...,...
86,2011-10-01,1,86
87,2011-10-01,2,41
88,2011-11-01,0,323
89,2011-11-01,1,36


Creating the table

In [9]:
cohort_table = pd.pivot(
    cohort_data, columns="CohortIndex", values="CustomerID", index="CohortMonth"
)
cohort_table

CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12
CohortMonth,,,,,,,,,,,,,
2010-12-01,885.0,324.0,286.0,340.0,321.0,352.0,321.0,309.0,313.0,350.0,331.0,445.0,235.0
2011-01-01,417.0,92.0,111.0,96.0,134.0,120.0,103.0,101.0,125.0,136.0,152.0,49.0,NaN
2011-02-01,380.0,71.0,71.0,108.0,103.0,94.0,96.0,106.0,94.0,116.0,26.0,NaN,NaN
2011-03-01,452.0,68.0,114.0,90.0,101.0,76.0,121.0,104.0,126.0,39.0,NaN,NaN,NaN
2011-04-01,300.0,64.0,61.0,63.0,59.0,68.0,65.0,78.0,22.0,NaN,NaN,NaN,NaN
2011-05-01,284.0,54.0,49.0,49.0,59.0,66.0,75.0,27.0,NaN,NaN,NaN,NaN,NaN
2011-06-01,242.0,42.0,38.0,64.0,56.0,81.0,23.0,NaN,NaN,NaN,NaN,NaN,NaN
2011-07-01,188.0,34.0,39.0,42.0,51.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08-01,169.0,35.0,42.0,41.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Creating the heatmap using Seaborn

Basic heatmap already shows some interesing information - last year's December attracted almost twice as many customers as any other month later on.

However, the current heatmap isn't very readable. Better approach to showing customer retention would be to use percentage instead of raw numbers.

In [ ]:
y_labels = cohort_table.index.strftime("%B %Y")

plt.figure(figsize=(16, 8))
sns.heatmap(cohort_table, cmap="Blues", yticklabels=y_labels, fmt="g", annot=True)
plt.title("Customer retention Heatmap")
plt.show()

#### Improving heatmap readability
Here I am dividing whole rows by the first element of each. This way I can measure retention in percentage of consumers who stayed.

In [11]:
cohort_table_ratio = cohort_table.divide(cohort_table.iloc[:, 0], axis=0)
cohort_table_ratio

CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12
CohortMonth,,,,,,,,,,,,,
2010-12-01,1.0,0.366102,0.323164,0.384181,0.362712,0.397740,0.362712,0.349153,0.353672,0.395480,0.374011,0.502825,0.265537
2011-01-01,1.0,0.220624,0.266187,0.230216,0.321343,0.287770,0.247002,0.242206,0.299760,0.326139,0.364508,0.117506,NaN
2011-02-01,1.0,0.186842,0.186842,0.284211,0.271053,0.247368,0.252632,0.278947,0.247368,0.305263,0.068421,NaN,NaN
2011-03-01,1.0,0.150442,0.252212,0.199115,0.223451,0.168142,0.267699,0.230088,0.278761,0.086283,NaN,NaN,NaN
2011-04-01,1.0,0.213333,0.203333,0.210000,0.196667,0.226667,0.216667,0.260000,0.073333,NaN,NaN,NaN,NaN
2011-05-01,1.0,0.190141,0.172535,0.172535,0.207746,0.232394,0.264085,0.095070,NaN,NaN,NaN,NaN,NaN
2011-06-01,1.0,0.173554,0.157025,0.264463,0.231405,0.334711,0.095041,NaN,NaN,NaN,NaN,NaN,NaN
2011-07-01,1.0,0.180851,0.207447,0.223404,0.271277,0.111702,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08-01,1.0,0.207101,0.248521,0.242604,0.124260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Readable heatmap
The heatmap now is much more readable, but now we lost the key information of an absolute number of customers acquired that month.

In [ ]:
y_labels = cohort_table_ratio.index.strftime("%B %Y")
plt.figure(figsize=(16, 8))
sns.heatmap(
    cohort_table_ratio.iloc[:, 1:],
    cmap="Blues",
    yticklabels=y_labels,
    annot=True,
    fmt=".0%",
)
plt.ylabel("First purchase month")
plt.xlabel("Months since the first purchase")
plt.title("Customer retention analysis (Cohorts)")
plt.show()

#### Adding the cohort size
To address that problem we need to create a subplot in which the absolute number of customers would be visible.

In [ ]:
cohort_size = (
    df_retention.groupby("CohortMonth")
    .agg({"CustomerID": "nunique"})
    .rename(columns={"CustomerID": "Cohort Size"})
)

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(16, 10), sharey=True, gridspec_kw={"width_ratios": [1, 14]}
)
sns.heatmap(cohort_size, cbar=False, cmap="Blues", annot=True, fmt="g", ax=ax1)
sns.heatmap(
    cohort_table_ratio.iloc[:, 1:],
    cmap="Blues",
    annot=True,
    fmt=".0%",
    ax=ax2,
    yticklabels=y_labels,
)
plt.subplots_adjust(wspace=0.02)
ax1.set_ylabel("First purchase month", fontsize=12)
ax2.set_ylabel("")
ax2.set_xlabel("Months since the first purchase", fontsize=12)
ax2.set_title("Customer retention analysis (Cohorts)")
plt.show()

### Executive Summary & Business Recommendations

#### Missing Data & Cohort Bias
* **Observation:** The great results of the December 2010 cohort are affected by missing past data. Because our dataset starts in this month, this group probably contains many old, loyal customers, not just the new ones. The sudden drop in December 2011 is also caused by incomplete data for that month.
* **Recommendation:** To correctly measure customer retention, we need to get historical data from before December 2010. This will help us separate the real "new" customers from the "existing" ones.

#### Seasonality & Marketing Budget
* **Observation:** There is a clear drop in new customer acquisition and their loyalty during the summer months. On the other hand, the Fall season (September-November) shows strong acquisition and high retention rates.
* **Recommendation:** The marketing department should adjust its budget. We should spend less money during the weak summer months and move this budget to September and October. This will help build a strong customer base before the winter holidays.

#### The "November Comeback"
* **Observation:** A diagonal pattern on the retention matrix shows that many inactive customers from previous months came back in November 2011. This was probably driven by Black Friday and early Christmas shopping.
* **Recommendation:** We should take advantage of this predictable trend. Product and Marketing teams should plan automated "win-back" email campaigns for inactive customers from Spring and Summer.


## RFM Analysis

In [14]:
df_monetary = df.groupby("CustomerID").agg(Monetary=("TotalPrice", "sum"))

ref_date = df_no_cancel["InvoiceDate"].max() + pd.Timedelta(days=1)
df_rf = (
    df_no_cancel.assign(
        RecencyDays=lambda x: (ref_date - x["InvoiceDate"]).dt.days,
    )
    .groupby("CustomerID")
    .agg(Recency=("RecencyDays", "min"), Frequency=("InvoiceNo", "nunique"))
)
df_rfm = df_rf.join(df_monetary)
df_rfm = df_rfm.set_index(df_rfm.index.astype("int64"))
df_rfm = df_rfm.loc[df_rfm["Monetary"] > 0].copy()
df_rfm.sample(5)

,Recency,Frequency,Monetary
CustomerID,,,
15445,264,1,113.50
17063,21,7,1421.59
12997,22,4,1197.94
17884,4,4,717.45
17172,79,1,313.26


Assign points to customers

In [15]:
labels = [1, 2, 3, 4, 5]
labels_reversed = labels.copy()
labels_reversed.reverse()

rfm_score = df_rfm.sort_values(
    by=["Monetary", "Frequency"], ascending=[True, True]
).assign(
    RecScore=lambda x: pd.qcut(
        x["Recency"].rank(method="first"), q=5, labels=labels_reversed
    ),
    FreqScore=lambda x: pd.qcut(
        x["Frequency"].rank(method="first"), q=5, labels=labels
    ),
    MonScore=lambda x: pd.qcut(x["Monetary"].rank(method="first"), q=5, labels=labels),
    RFMScore=lambda x: (
        x["RecScore"].astype("str")
        + "-"
        + x["FreqScore"].astype("str")
        + "-"
        + x["MonScore"].astype("str")
    ),
)
rfm_score

,Recency,Frequency,Monetary,RecScore,FreqScore,MonScore,RFMScore
CustomerID,,,,,,,
18274,30,1,1.776357e-15,4,1,1,4-1-1
12607,60,1,3.552714e-15,3,1,1,3-1-1
13762,219,1,3.552714e-15,1,1,1,1-1-1
12558,8,1,1.065814e-14,5,1,1,5-1-1
12454,56,1,5.684342e-14,3,1,1,3-1-1
...,...,...,...,...,...,...,...
12415,24,21,1.237254e+05,4,5,5,4-5-5
14911,1,201,1.325726e+05,5,5,5,5-5-5
17450,8,46,1.874822e+05,5,5,5,5-5-5


In [ ]:
r = rfm_score["RecScore"].astype(int)
f = rfm_score["FreqScore"].astype(int)
m = rfm_score["MonScore"].astype(int)

conditions = [
    (r + f + m) >= 14,
    (r <= 2) & (f >= 4),
    (r >= 4) & (f <= 2),
    (r <= 2) & (m <= 2),
]
choices = ["Champion", "At risk", "Newcomer", "Hibernating"]
rfm_score = rfm_score.assign(
    CustomerSegment=pd.Categorical(np.select(conditions, choices, default="Standard"))
)
sns.barplot(rfm_score.groupby("CustomerSegment").size())
plt.show()

Pareto rule in gross revenue per customer

In [17]:
df_pareto = (
    df.groupby("CustomerID")
    .agg(TotalSpending=("TotalPrice", "sum"))
    .sort_values(by="TotalSpending", ascending=False)
    .assign(
        SpendingCumSum=lambda x: x["TotalSpending"].cumsum(),
        RevenuePercent=lambda x: x["SpendingCumSum"] / x["TotalSpending"].sum(),
    )
)

top_80p_revenue = df_pareto[df_pareto["RevenuePercent"] <= 0.80]

top_count = len(top_80p_revenue)
total_count = len(df_pareto)

ratio = top_count / total_count

f"80% of revenue is generated by top {ratio:.1%} customers"

'80% of revenue is generated by top 26.8% customers'

### Hourly buying time in span of a week

In [ ]:
days_dict = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}
days_order = np.arange(0, 7)
summary_weekly_hourly = (
    df.assign(
        Day=lambda x: x["InvoiceDate"].dt.day_of_week,
        Hour=lambda x: x["InvoiceDate"].dt.hour,
    )
    .groupby(["Day", "Hour"])
    .agg(InvoiceCount=("InvoiceNo", "nunique"))
    .reset_index()
)

weekly_purchase_matrix = (
    summary_weekly_hourly.pivot(index="Day", columns="Hour", values="InvoiceCount")
    .reindex(days_order)
    .fillna(0)
)

golden_hour_row = summary_weekly_hourly.loc[
    summary_weekly_hourly["InvoiceCount"].idxmax()
]

print(
    f"Our golden hour is at {days_dict[golden_hour_row['Day']]}, {golden_hour_row['Hour']}:00. {golden_hour_row['InvoiceCount']} transactions were made during that hour."
)

plt.figure(figsize=(16, 8))
ax = sns.heatmap(weekly_purchase_matrix, annot=True, cmap="YlGnBu", fmt="g")
ax.set_yticklabels(days_dict.values(), rotation=0)
plt.title("When to send the push? (Purchases by hours and days of week)")
plt.ylabel("Day of week", fontsize=12)
plt.xlabel("Hour of purchase", fontsize=12)
plt.show()

### Products often bought in pairs

In [19]:
from itertools import combinations
from collections import Counter

product_dict = (
    df_no_cancel.drop_duplicates(subset=["StockCode"])
    .set_index("StockCode")["Description"]
    .to_dict()
)

transactions = df_no_cancel.groupby("InvoiceNo")["StockCode"].unique()

pair_counter = Counter()

for basket in transactions:
    if len(basket) > 1:
        pair_counter.update(combinations(sorted(basket), 2))
top_pairs = pair_counter.most_common(20)

pd.DataFrame(
    data=(
        (pair[0], product_dict.get(pair[0]), pair[1], product_dict.get(pair[1]), count)
        for pair, count in top_pairs
    ),
    columns=["StockCodeA", "ProductNameA", "StockCodeB", "ProductNameB", "PairCount"],
)

,StockCodeA,ProductNameA,StockCodeB,ProductNameB,PairCount
0,22386,JUMBO BAG PINK POLKADOT,85099B,JUMBO BAG RED RETROSPOT,546
1,22697,GREEN REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,541
2,22726,ALARM CLOCK BAKELIKE GREEN,22727,ALARM CLOCK BAKELIKE RED,530
3,20725,LUNCH BAG RED RETROSPOT,22384,LUNCH BAG PINK POLKADOT,523
4,20725,LUNCH BAG RED RETROSPOT,22383,LUNCH BAG SUKI DESIGN,519
5,20725,LUNCH BAG RED RETROSPOT,20727,LUNCH BAG BLACK SKULL.,517
6,82482,WOODEN PICTURE FRAME WHITE FINISH,82494L,WOODEN FRAME ANTIQUE WHITE,468
7,23203,JUMBO BAG DOILEY PATTERNS,85099B,JUMBO BAG RED RETROSPOT,468
8,20725,LUNCH BAG RED RETROSPOT,22382,LUNCH BAG SPACEBOY DESIGN,467
9,20727,LUNCH BAG BLACK SKULL.,22383,LUNCH BAG SUKI DESIGN,465


Top Countries by average revenue per user

In [20]:
arpu_summary = (
    df.loc[df["Country"] != "United Kingdom"]
    .groupby(["Country", "CustomerID"], as_index=False)
    .agg(TotalSpending=("TotalPrice", "sum"))
    .groupby("Country")
    .filter(lambda x: len(x) >= 20)
    .groupby("Country")
    .agg(
        AverageRevenuePerUser=("TotalSpending", "mean"),
        UniqueCustomers=("CustomerID", "nunique"),
    )
    .sort_values(by="AverageRevenuePerUser", ascending=False)
    .round(2)
)

arpu_summary

,AverageRevenuePerUser,UniqueCustomers
Country,,
Switzerland,2654.26,21
Germany,2333.67,95
France,2261.07,87
Spain,1766.92,31
Belgium,1636.44,25


Churn analysis

In [88]:
customer_churn_status = (
    df_no_cancel.groupby("CustomerID")
    .agg(LastPurchase=("InvoiceDate", "max"))
    .assign(
        Recency=lambda x: (ref_date - x["LastPurchase"]).dt.days,
        Status=lambda x: np.where(x["Recency"] <= 90, "Active", "Churned"),
    )
)
customer_churn_status.index = customer_churn_status.index.astype("int64")
display(
    (customer_churn_status["Status"].value_counts(normalize=True) * 100).round(2),
    customer_churn_status.sample(5),
)

Status
Active     66.6
Churned    33.4
Name: proportion, dtype: float64

,LastPurchase,Recency,Status
CustomerID,,,
17279,2011-10-13 11:58:00,58,Active
15990,2011-10-26 15:58:00,44,Active
15524,2011-11-15 12:09:00,25,Active
15075,2011-09-23 12:27:00,78,Active
15939,2011-09-11 11:30:00,90,Active


Median monthly spend per customer

In [ ]:
# assigning cohorts on first positive transaction, but still counting cancellations in spending
cohort_map = (
    df_no_cancel.assign(
        InvoiceMonth=df_no_cancel["InvoiceDate"].values.astype("datetime64[M]")
    )
    .groupby("CustomerID")
    .agg(CohortMonth=("InvoiceMonth", "min"))
)

aov_heatmap_data = (
    df.assign(InvoiceMonth=lambda x: x["InvoiceDate"].values.astype("datetime64[M]"))
    .merge(cohort_map, on="CustomerID")
    .query("InvoiceMonth>=CohortMonth")
    .assign(
        CohortIndex=lambda x: (
            (x["InvoiceMonth"].dt.year - x["CohortMonth"].dt.year) * 12
            + (x["InvoiceMonth"].dt.month - x["CohortMonth"].dt.month)
        )
    )
    .groupby(["CohortMonth", "CohortIndex", "CustomerID"], as_index=False)
    .agg(SpendingInMonth=("TotalPrice", "sum"))
    .pivot_table(
        columns="CohortIndex",
        index="CohortMonth",
        values="SpendingInMonth",
        aggfunc="median",
    )
)

plt.figure(figsize=(21, 10))
sns.heatmap(aov_heatmap_data, annot=True, cmap="Blues", fmt=".0f", yticklabels=y_labels)
plt.title("Monthly Median Spending (Cohort Analysis)", fontsize=16)
plt.ylabel("First Purchase Month", fontsize=12)
plt.xlabel("Months Since First Purchase", fontsize=12)
plt.show()

In [ ]:
cohort_ltv_summary = (
    df.assign(InvoiceMonth=lambda x: x["InvoiceDate"].values.astype("datetime64[M]"))
    .merge(cohort_map, on="CustomerID")
    .query("InvoiceMonth >= CohortMonth")
    .groupby(["CohortMonth", "CustomerID"], as_index=False)
    .agg(TotalLifetimeSpend=("TotalPrice", "sum"))
    .groupby("CohortMonth")
    .agg(
        MedianLTV=("TotalLifetimeSpend", "median"),
        CustomersInCohort=("CustomerID", "nunique"),
    )
    .reset_index()
)

cohort_ltv_summary

,CohortMonth,MedianLTV,CustomersInCohort
0,2010-12-01,1641.760,885
1,2011-01-01,1126.850,417
2,2011-02-01,830.515,380
3,2011-03-01,686.945,452
4,2011-04-01,660.700,300
5,2011-05-01,620.575,284
6,2011-06-01,508.045,242
7,2011-07-01,488.695,188
8,2011-08-01,519.610,169
9,2011-09-01,513.200,299
